# mmWave Radar Sensing: Hybrid PO

This tutorial combines physical optics (PO) for a moving human with ray tracing (RT) for a static bedroom. The calibrated hybrid pipeline returns four coherent signal components:

```text
total = H_direct + E_static + H_env + E_human
```

`H_direct` is direct human PO, `E_static` is the blocked static-environment response, and `H_env`/`E_human` are the two human-environment coupling directions. The notebook focuses on configuring the public API and viewing the resulting radar products.


In [ ]:
# Resolve the local src/ tree and keep Matplotlib's cache outside the repository.
import os
import sys
import tempfile
import time

from pathlib import Path

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "src" / "mmWaveRadar").exists():
    repo_root = repo_root.parent
if not (repo_root / "src" / "mmWaveRadar").exists():
    raise RuntimeError(
        "Run this notebook from inside a HERMES source checkout, "
        "including the top-level demo/ directory."
    )
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

os.environ.setdefault(
    "MPLCONFIGDIR",
    str(Path(tempfile.gettempdir()) / "mmwave-radar-mpl"),
)

import matplotlib.pyplot as plt
import numpy as np

from mmWaveRadar import amass_to_smpl_motion_sequence
from mmWaveRadar.dsp import (
    plot_axis_image,
    range_doppler_map,
    range_fft,
    range_profile_from_cube,
)
from mmWaveRadar.materials import human_skin_material
from mmWaveRadar.radar import FMCWConfig, RadarSensor
from mmWaveRadar.simulation import (
    db_relative,
    run_calibrated_hybrid_po,
    select_torch_device,
)
from mmWaveRadar.targets import MeshTarget
from mmWaveRadar.tutorial_support import (
    load_bedroom_scene,
    plot_bedroom_scene_projection,
    plot_mesh_projection,
    radar_pose_in_front_of_chest,
    time_window_mesh_sequence,
)


## Radar Configuration

The TI board preset supplies the antenna geometry and transmitter count. One radar frame provides 64 chirps for the range-Doppler view while keeping the tutorial quick to run.


In [ ]:
fmcw = FMCWConfig(
    carrier_frequency=60e9,
    slope=68e12,
    chirp_duration=58e-6,
    chirp_repetition_time=65e-6,
    sampling_frequency=4.5e6,
    num_adc_samples=225,
    num_chirps_per_frame=64,
    frame_period=50e-3,
)
ti_board_model = "IWR6843AOPEVM"
ti_pattern_mode = "cosine30"
num_frames = 1
simulation_duration_s = num_frames * fmcw.frame_period

print(f"Wavelength: {1e3 * fmcw.wavelength:.2f} mm")
print(
    f"Run length: {num_frames} frame x {fmcw.num_chirps_per_frame} chirps "
    f"({simulation_duration_s:.3f} s radar interval)"
)


## AMASS Motion Mesh

The repository includes the CMU walking motion in AMASS-style base-SMPL format. SMPL model files must be obtained separately and placed under `models/smpl_models/`, or supplied with `MMWAVE_SMPL_MODEL_DIR`. The selected source window is rebased to radar time zero.


In [ ]:
dataset_root = Path(
    os.environ.get("MMWAVE_DATASET_ROOT", repo_root / "data")
).expanduser()
amass_npz_path = Path(
    os.environ.get(
        "MMWAVE_AMASS_NPZ",
        dataset_root / "AMASS" / "walking_poses_cmu_105_02.npz",
    )
).expanduser()
smpl_model_dir = Path(
    os.environ.get("MMWAVE_SMPL_MODEL_DIR", repo_root / "models" / "smpl_models")
).expanduser()

missing_paths = [
    path for path in (amass_npz_path, smpl_model_dir) if not path.exists()
]
if missing_paths:
    missing = "\n".join(f"  - {path}" for path in missing_paths)
    raise FileNotFoundError(
        "Hybrid-PO.ipynb requires these AMASS/SMPL inputs:\n"
        f"{missing}\n"
        "Obtain the licensed SMPL model separately and set "
        "MMWAVE_SMPL_MODEL_DIR if needed."
    )

smpl_device = select_torch_device()
raw_mesh_sequence = amass_to_smpl_motion_sequence(
    str(amass_npz_path),
    str(smpl_model_dir),
    model_type="smpl",
    device=smpl_device,
)

# Start where subject 105 begins walking; retain one second for the preview.
motion_start_time_s = 3.0
mesh_preview_duration_s = 1.0
mesh_sequence = time_window_mesh_sequence(
    raw_mesh_sequence,
    start_time_s=motion_start_time_s,
    duration_s=max(simulation_duration_s, mesh_preview_duration_s),
)

radar_position, radar_orientation, chest_front_point, _ = (
    radar_pose_in_front_of_chest(mesh_sequence, clearance_m=0.50)
)
radar = RadarSensor.from_ti_board(
    ti_board_model,
    fmcw=fmcw,
    position=tuple(float(value) for value in radar_position),
    orientation=tuple(float(value) for value in radar_orientation),
    pattern_mode=ti_pattern_mode,
)
fmcw = radar.fmcw
target = MeshTarget(
    name="amass_human_hybrid_po",
    mesh_sequence=mesh_sequence,
    material=human_skin_material("human-hybrid-po-skin"),
)

last_chirp_time_s = fmcw.chirp_time(
    num_frames - 1, fmcw.num_chirps_per_frame - 1
)
if last_chirp_time_s > float(mesh_sequence.times[-1]) + 1e-9:
    raise ValueError("The selected mesh window does not cover the last radar chirp.")

print(f"Motion: {amass_npz_path}")
print(f"SMPL model: {smpl_model_dir} ({smpl_device})")
print(
    f"Mesh: {mesh_sequence.vertex_count} vertices, "
    f"{mesh_sequence.faces.shape[0]} faces"
)
print(
    f"Radar: {radar.hardware.name}, {radar.hardware.num_tx} Tx, "
    f"{radar.hardware.num_rx} Rx"
)


### Human Mesh in the Bedroom

The columns show the beginning, middle, and end of a one-second motion preview. The green marker appears only at the initial pose because it is the radar's initial aim point, not a tracked landmark.

The bedroom geometry and radio materials are defined by `BEDROOM_SCENE_XML` in `src/mmWaveRadar/tutorial_support.py`. When changing that scene, also update `BEDROOM_SCENE_BOXES` there so this 2D preview remains consistent with the traced geometry.


In [ ]:
mesh_preview_times_s = np.linspace(
    float(mesh_sequence.times[0]),
    min(mesh_preview_duration_s, float(mesh_sequence.times[-1])),
    3,
)

fig, axes = plt.subplots(2, 3, figsize=(14, 7), constrained_layout=True)
for column, time_s in enumerate(mesh_preview_times_s):
    plot_mesh_projection(
        axes[0, column],
        mesh_sequence,
        time_s,
        radar.position,
        f"Human mesh x-y, t={time_s:.3f} s",
        axes=(0, 1),
    )
    plot_bedroom_scene_projection(axes[0, column], axes=(0, 1))

    plot_mesh_projection(
        axes[1, column],
        mesh_sequence,
        time_s,
        radar.position,
        f"Human mesh x-z, t={time_s:.3f} s",
        axes=(0, 2),
    )
    plot_bedroom_scene_projection(axes[1, column], axes=(0, 2))

plt.show()


## Run the Calibrated Hybrid Pipeline

These are the main tutorial controls. `coupling_enabled` includes the two human-environment terms, while `max_depth` and the RT sample limits control the static and calibration ray tracing. A value of zero for `coupling_max_reflectors` keeps all eligible single-reflection surfaces.


In [ ]:
coupling_enabled = True
coupling_max_reflectors = 0
po_visibility_samples_per_face = 4
max_depth = 2
rt_samples_per_src = 80_000
rt_max_num_paths_per_src = 80_000
calibration_frame_range = (0, 1)

started = time.perf_counter()
hybrid_run = run_calibrated_hybrid_po(
    scene=load_bedroom_scene(merge_shapes=False),
    coupling_scene=load_bedroom_scene(merge_shapes=False),
    radar=radar,
    target=target,
    num_frames=num_frames,
    calibration_frame_range=calibration_frame_range,
    po_visibility_samples_per_face=po_visibility_samples_per_face,
    coupling_enabled=coupling_enabled,
    coupling_max_reflectors=coupling_max_reflectors,
    rt_samples_per_src=rt_samples_per_src,
    rt_max_num_paths_per_src=rt_max_num_paths_per_src,
    max_depth=max_depth,
    progress=False,
    print_reflector_summary=False,
)
elapsed_s = time.perf_counter() - started
hybrid_cube = hybrid_run.cube

required_components = {
    "human_po",
    "static_environment_blocked",
    "human_env",
    "env_human",
}
if hybrid_cube.components is None:
    raise RuntimeError("The hybrid result does not contain component ADC arrays.")
missing_components = required_components.difference(hybrid_cube.components)
if missing_components:
    raise RuntimeError(f"Missing hybrid components: {sorted(missing_components)}")

print(f"Hybrid cube: {hybrid_cube.adc.shape}")
print(f"Elapsed time: {elapsed_s:.2f} s")
print(f"Coupling reflectors: {hybrid_run.reflector_count}")


## Component Radar Products

The plots compare the four components and their coherent total using one shared dB reference. Range profiles average the available chirps; range-Doppler maps use the first radar frame.


In [ ]:
component_results = [
    ("H_direct", hybrid_cube.components["human_po"]),
    ("E_static", hybrid_cube.components["static_environment_blocked"]),
    ("H_env", hybrid_cube.components["human_env"]),
    ("E_human", hybrid_cube.components["env_human"]),
    ("Total", hybrid_cube.adc),
]
range_limit_m = 6.0
display_floor_db = -80.0


def extract_products(adc):
    adc = np.asarray(adc)
    chirps = adc.reshape((-1, adc.shape[-2], adc.shape[-1]))
    range_cube, profile_ranges = range_fft(
        chirps, fmcw=fmcw, window="hann", nfft_mult=4
    )
    profile = range_profile_from_cube(
        range_cube, combine_antennas="sum_power", combine_chirps="mean"
    )
    rd_cube, rd_ranges, velocities = range_doppler_map(
        adc[0],
        fmcw=fmcw,
        num_tx=radar.hardware.num_tx,
        win_range="hann",
        win_doppler="hann",
    )
    rd_power = np.sum(np.abs(rd_cube) ** 2, axis=-1)
    return profile, profile_ranges, rd_power, rd_ranges, velocities


products = [
    (label, extract_products(adc)) for label, adc in component_results
]
profile_ranges = products[0][1][1]
profile_mask = profile_ranges <= range_limit_m
profiles_db = db_relative(np.stack([
    product[0][profile_mask] for _, product in products
]))
rd_ranges = products[0][1][3]
rd_mask = rd_ranges <= range_limit_m
rd_maps_db = db_relative(np.stack([
    product[2][:, rd_mask] for _, product in products
]))

fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
profile_ax = axes[0, 0]
for (label, _), profile_db in zip(products, profiles_db):
    profile_ax.plot(profile_ranges[profile_mask], profile_db, label=label)
profile_ax.set(
    xlim=(0, range_limit_m),
    ylim=(display_floor_db, 3),
    xlabel="range [m]",
    ylabel="power [dB, shared reference]",
    title="Mean range profile",
)
profile_ax.grid(True, alpha=0.3)
profile_ax.legend(fontsize=8)

rd_axes = [axes[0, 1], axes[0, 2], *axes[1]]
for ax, (label, product), rd_map_db in zip(rd_axes, products, rd_maps_db):
    image = plot_axis_image(
        rd_map_db,
        product[3][rd_mask],
        product[4],
        ax=ax,
        xlabel="range [m]",
        ylabel="velocity [m/s]",
        title=label,
        cmap="magma",
        vmin=display_floor_db,
        vmax=0,
    )
fig.colorbar(image, ax=rd_axes, label="power [dB, shared reference]")
fig.suptitle("Calibrated Hybrid PO components")
plt.show()
